# IMPORTS

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt
import time
import os
import copy
from torch.utils import data
from PIL import Image
import random
import cv2
import json
import datetime
import multiprocessing
import queue
import threading
import lightning.pytorch as pl

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FasterRCNN_ResNet50_FPN_Weights # ADDED THIS LINE

from torchvision.datasets import CocoDetection
from torch.utils.data import DataLoader, Dataset

from pycocotools.coco import COCO

import os

to_tensor = torchvision.transforms.ToTensor()

print("PyTorch Version: ",torch.__version__)
print("Torchvision Version: ",torchvision.__version__)

PyTorch Version:  2.9.1+rocmsdk20251207
Torchvision Version:  0.24.0+rocmsdk20251207


In [ ]:
# Generates training images
!python generate.py 32

Starting training image generation...

0/32 training images generated

20/32 training images generated
Finished generating training images
Starting validation image generation...

0/8 validation images generated
Finished generating validation images
Creating COCO file...
COCO file created
Finished :)


In [2]:
# Custom PyTorch Dataset to load COCO-format annotations and images
class CocoDetectionDataset(Dataset):
    # Init function: loads annotation file and prepares list of image IDs
    def __init__(self, image_dir, annotation_path, transforms=None):
        self.image_dir = image_dir
        self.coco = COCO(annotation_path)
        self.image_ids = list(self.coco.imgs.keys())
        self.transforms = transforms

    # Returns total number of images
    def __len__(self):
        return len(self.image_ids)

    # Fetches a single image and its annotations
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.coco.loadImgs(image_id)[0]
        image_path = os.path.join(self.image_dir, image_info['file_name'])
        image = Image.open(image_path).convert("RGB")

        # Load all annotations for this image
        annotation_ids = self.coco.getAnnIds(imgIds=image_id)
        annotations = self.coco.loadAnns(annotation_ids)

        # Extract bounding boxes and labels from annotations
        boxes = []
        labels = []
        for obj in annotations:
            xmin, ymin, width, height = obj['bbox']
            xmax = xmin + width
            ymax = ymin + height
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(obj['category_id'])

        # Convert annotations to PyTorch tensors
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        area = torch.as_tensor([obj['area'] for obj in annotations], dtype=torch.float32)
        iscrowd = torch.as_tensor([obj.get('iscrowd', 0) for obj in annotations], dtype=torch.int64)

        # Package everything into a target dictionary
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }

        # Apply transforms if any were passed
        if self.transforms:
            image = self.transforms(image)

        return image, target

In [3]:
class RCNNDataModule(pl.LightningDataModule):
    def __init__(self,
                 train_image_dir,
                 train_annotation_path,
                 val_image_dir,
                 val_annotation_path,
                 batch_size=4,
                 num_workers=4,
                 **kwargs):
        super().__init__(**kwargs)
        self.train_image_dir = train_image_dir
        self.train_annotation_path = train_annotation_path
        self.val_image_dir = val_image_dir
        self.val_annotation_path = val_annotation_path
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.data_train = None
        self.data_val = None

    def setup(self, stage: str):
        if (stage == 'fit' or stage == 'validate' and not(self.data_train and self.data_val)):
            # Load up the images
            self.data_train = CocoDetectionDataset(image_dir=self.train_image_dir,
                                                   annotation_path=self.train_annotation_path,
                                                   transforms=transforms.ToTensor())

            self.data_val = CocoDetectionDataset(image_dir=self.val_image_dir,
                                                 annotation_path=self.val_annotation_path,
                                                 transforms=transforms.ToTensor())

    def collate_fn(self, x):
        return tuple(zip(*x))

    def train_dataloader(self):
        return torch.utils.data.DataLoader(self.data_train,
                                           batch_size=self.batch_size,
                                           num_workers=self.num_workers,
                                           shuffle=True,
                                           collate_fn=self.collate_fn)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(self.data_val,
                                           batch_size=self.batch_size,
                                           num_workers=self.num_workers,
                                           shuffle=False,
                                           collate_fn=self.collate_fn)

In [ ]:
# The actual network
class RCNN(pl.LightningModule):
    def __init__(self, 
                 num_classes: int,
                 **kwargs):
        super().__init__()

        self.rcnn = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights)

        in_features = self.rcnn.roi_heads.box_predictor.cls_score.in_features
        self.rcnn.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    def forward(self, input_images):
      y = self.rcnn(input_images)
      return y

    def forward(self, input_images, input_data):
        y = self.rcnn(input_images, input_data)
        return y

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=0.001)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

        return [optimizer], [scheduler]

    def training_step(self, train_batch, batch_idx):
        imgs, annotations_batch = train_batch
        input_images = [img.to(self.device) for img in imgs]
        input_data = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in annotations_batch]

        self.train()
        loss_dict = self(input_images, input_data)
        loss = sum(loss for loss in loss_dict.values())

        self.log('train_loss', loss, on_step=False, on_epoch=True)

        return loss

    def validation_step(self, val_batch, batch_idx):
        imgs, annotations_batch = val_batch
        input_images = [img.to(self.device) for img in imgs]
        input_data = [{k: v.to(self.device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in annotations_batch]

        self.train()
        loss_dict = self(input_images, input_data)
        loss = sum(loss for loss in loss_dict.values())

        self.log('val_loss', loss, on_step=False, on_epoch=True)
        
        return loss

        

In [5]:
data_module = RCNNDataModule(train_image_dir="dataset/images/train",
                             train_annotation_path="dataset/instances_train.json",
                             val_image_dir="dataset/images/val",
                             val_annotation_path="dataset/instances_val.json")

In [6]:
model = RCNN(num_classes=23, device="cuda")

c:\Users\Philip\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
trainer = pl.Trainer(logger=None,
                     max_epochs=1,
                     enable_progress_bar=True,
                     log_every_n_steps=0,
                     callbacks=[pl.callbacks.RichProgressBar(refresh_rate=20)])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [ ]:
trainer.fit(model, data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('AMD Radeon RX 6800') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


┏━━━┳━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ rcnn │ FasterRCNN │ 41.4 M │ train │     0 │
└───┴──────┴────────────┴────────┴───────┴───────┘

Trainable params: 41.2 M                                                                                           
Non-trainable params: 222 K                                                                                        
Total params: 41.4 M                                                                                               
Total estimated model params size (MB): 165                                                                        
Modules in train mode: 189                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\Philip\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\lightning\pytorch\trainer\connectors\data_conne
ctor.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker 
initialization.

# results

In [1]:
img_path = "dataset/images/train/0001.png"
image_bgr = cv2.imread(img_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
image_pil = Image.fromarray(image_rgb)

NameError: name 'cv2' is not defined

In [ ]:
# Transform image
transform = transforms.Compose([transforms.ToTensor()])
image_tensor = transform(image_pil).unsqueeze(0)

In [ ]:
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    predictions = model(image_tensor.to(device))

In [ ]:
model.eval()  # Set the model to evaluation mode
with torch.no_grad():
    predictions = model(image_tensor.to(device))

In [ ]:
def show_prediction(prediction, image_bgr):
  boxes = prediction['boxes']
  labels = prediction['labels']
  scores = prediction['scores']

  print(f"Raw scores: {scores}")
  print(f"Raw labels: {labels}")

  #label_list = [f"Agent_{i}" for i in range(1, 23)] # Assuming class IDs are 1-22 for agents
  label_list = ["Chamber",
              "Cypher",
              "Yoru",
              "Killjoy",
              "KAYO",
              "Gekko",
              "Astra",
              "Deadlock",
              "Breach",
              "Omen",
              "Reyna",
              "Phoenix",
              "Tejo",
              "Raze",
              "Brimstone",
              "Vyse",
              "Fade",
              "Sova",
              "Jett",
              "Neon",
              "Skye",
              "Viper",
              "Sage"]

  # Threshold
  threshold = 0.5

  # Keep track of whether any boxes were drawn
  b_boxes_drawn = False

  for i in range(len(boxes)):
      if scores[i] > threshold:
          box = boxes[i].cpu().numpy().astype(int)
          label_id = labels[i].item() # Get integer value of label
          # Ensure label_id is within the valid range for label_list
          if 0 < label_id <= len(label_list):
              label = label_list[label_id] # Adjust index if label_list is 0-indexed
          else:
              label = f"Unknown ({label_id})"

          score = scores[i].item()

          # draw label and score
          text = f"{label}: {score:.2f}"
          cv2.putText(image_bgr, text, (box[0], box[1] - 10), cv2.FONT_HERSHEY_SIMPLEX,
                      0.4, (0, 0, 0), 3, cv2.LINE_AA)
          cv2.putText(image_bgr, text, (box[0], box[1] - 10), cv2.FONT_HERSHEY_SIMPLEX,
                      0.4, (0, 255, 0), 2, cv2.LINE_AA)

          # Draw rectangle and label
          cv2.rectangle(image_bgr, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
          b_boxes_drawn = True

  # Check if any bounding boxes were drawn, if not, indicate it
  if not b_boxes_drawn:
      print(f"No bounding boxes detected above threshold {threshold}.")

  # Convert BGR to RGB for correct display with matplotlib
  image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

  # Show image with larger figure size
  plt.figure(figsize=(16, 12))  # Increase size as needed
  plt.imshow(image_rgb)
  plt.axis('off')
  plt.show()

In [ ]:
def make_prediction(model, img_path):
  image_bgr = cv2.imread(img_path)
  image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
  image_pil = Image.fromarray(image_rgb)

  # Transform image
  transform = transforms.Compose([transforms.ToTensor()])
  image_tensor = transform(image_pil).unsqueeze(0)

  model.eval()  # Set the model to evaluation mode
  with torch.no_grad():
      predictions = model(image_tensor.to(device))

  show_prediction(predictions[0], image_bgr)

In [ ]:
images = ["snippet.png", "snippet2.png", "snippet3.png", "snippet4.png", "snippet5.png"]
for image in images:
  make_prediction(model=model, img_path=image)